# 03 - Synthesis: từ unitary tới chuỗi cổng, và bài toán T-count

Notebook `01` và `02` lo chuyện **ở đâu** (qubit nào, đường nào). Notebook này lo chuyện **cổng gì**:
cho một unitary $U$, tìm chuỗi cổng ngắn nhất trong basis gates thực hiện được $U$.

Có **hai** bài toán synthesis khác nhau về bản chất:

| | Gate set | Bài toán | Kết quả |
|---|---|---|---|
| **Exact synthesis** | liên tục (`rz(θ)`, `cx`) | phân tích chính xác | hữu hạn cổng, sai số 0 |
| **Approximate synthesis** | rời rạc (Clifford+T) | xấp xỉ tới $\epsilon$ | số cổng tăng theo $\log(1/\epsilon)$ |

Loại thứ nhất là chuyện của máy NISQ hôm nay. Loại thứ hai là chuyện của máy **fault-tolerant**:
mã sửa lỗi chỉ thực hiện được một tập cổng rời rạc, và trong tập đó cổng $T$ đắt hơn mọi cổng khác
hàng bậc. Nên **T-count** trở thành hàm cost chính - và đó là bài toán mà hướng RL kiểu
AlphaTensor-Quantum nhắm vào.

Notebook này cũng chứa **hai cái bẫy** nữa, và cái thứ nhất là loại nguy hiểm nhất: nó làm bạn in ra
một bảng số liệu trông hợp lý mà hoàn toàn vô nghĩa.

In [ ]:
import numpy as np
import pandas as pd

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import CXGate, CZGate, RZGate, SwapGate, UnitaryGate, iSwapGate
from qiskit.circuit.random import random_circuit
from qiskit.quantum_info import Operator, random_unitary
from qiskit.synthesis import OneQubitEulerDecomposer, TwoQubitBasisDecomposer
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import (
    Collect2qBlocks,
    ConsolidateBlocks,
    OptimizeCliffordT,
    SolovayKitaev,
    UnitarySynthesis,
)

CLIFFORD_T = ["h", "t", "tdg", "s", "sdg", "x", "z"]


def dem_T(qc):
    ops = qc.count_ops()
    return ops.get("t", 0) + ops.get("tdg", 0)


def sai_so(qc, U):
    """Khoảng cách toán tử tới U, đã bỏ global phase."""
    V = Operator(qc).data
    pha = np.vdot(V.flatten(), U.flatten())
    pha /= abs(pha)
    return np.linalg.norm(V * np.conj(pha) - U, 2)


print("Clifford+T gate set:", CLIFFORD_T)

## 1. Exact synthesis 1 qubit: ba góc Euler là đủ

Mọi unitary $2 \times 2$ viết được dưới dạng ba phép quay quanh hai trục:

$$U = e^{i\alpha}\, R_z(\phi)\, R_y(\theta)\, R_z(\lambda)$$

Ba tham số thực (cộng global phase) khớp đúng với 3 bậc tự do của $SU(2)$, nên **ba cổng quay là
tối thiểu và luôn đủ**. Phần cứng siêu dẫn thường cho `rz` gần như miễn phí (chỉ là đổi pha của
xung điều khiển) nhưng `sx` thì tốn, nên basis `ZSX` dùng 3 `rz` + 2 `sx`.

In [ ]:
U1 = random_unitary(2, seed=1)
print(f"{'basis':6s} {'cổng tổng hợp ra':34s} {'sai số':>9s}")
for basis in ["ZYZ", "U", "ZSX", "RR"]:
    c = OneQubitEulerDecomposer(basis=basis)(U1)
    print(f"{basis:6s} {str(dict(c.count_ops())):34s} {sai_so(c, U1.data):9.1e}")
print("\n-> cả bốn đều chính xác (sai số ~ 1e-16 là lỗi làm tròn số thực).")
print("-> khác nhau chỉ ở chỗ cổng nào thiết bị chạy được và cổng nào đắt.")

## 2. Exact synthesis 2 qubit: định lý KAK và con số 3

Với unitary $4 \times 4$ thì kết quả mạnh hơn nhiều, và là một trong những định lý hữu dụng nhất của
compilation. Phân tích **KAK** (Cartan) nói mọi $U \in SU(4)$ viết được thành

$$U = (A_1 \otimes A_2)\; e^{\,i(c_1 XX + c_2 YY + c_3 ZZ)}\; (B_1 \otimes B_2)$$

với $A_i, B_i$ là unitary 1 qubit. Ba hệ số $(c_1, c_2, c_3)$ - gọi là **toạ độ Weyl** - quyết định
hoàn toàn cần bao nhiêu cổng `cx`, và câu trả lời chỉ có **bốn** khả năng:

| Số `cx` cần | Loại unitary |
|---|---|
| 0 | tích tensor, không có entanglement |
| 1 | `cx`, `cz`, `cphase` |
| 2 | `iswap`, `sqrt(SWAP)` |
| **3** | `swap`, và **hầu hết** unitary ngẫu nhiên |

Nghĩa là **3 `cx` luôn đủ** cho bất kỳ unitary 2 qubit nào. `num_basis_gates()` cho biết con số đó
trước khi tổng hợp - nó chỉ tính toạ độ Weyl, rất nhanh.

In [ ]:
tich = QuantumCircuit(2)
tich.h(0)
tich.t(1)

dec = TwoQubitBasisDecomposer(CXGate())
print(f"{'unitary':22s} {'cx tối thiểu':>13s} {'cx tổng hợp':>12s} {'chính xác':>10s}")
for ten, U in [
    ("H ⊗ T (tích tensor)", Operator(tich)),
    ("CX", Operator(CXGate())),
    ("CZ", Operator(CZGate())),
    ("iSWAP", Operator(iSwapGate())),
    ("SWAP", Operator(SwapGate())),
    ("ngẫu nhiên", random_unitary(4, seed=5)),
]:
    c = dec(U)
    print(f"{ten:22s} {dec.num_basis_gates(U):13d} {c.count_ops().get('cx', 0):12d} "
          f"{str(Operator(c).equiv(U)):>10s}")

print("\n-> SWAP cần 3 cx. Đó chính là cái giá của routing ở notebook 02:")
print("   mỗi SWAP chèn vào là 3 cổng 2-qubit thật sự chạy trên thiết bị.")

### Synthesis như một chiến lược tối ưu

KAK không chỉ để phân tích một unitary cho sẵn. Nó là một **cách tối ưu mạch**: gom một khối cổng
2-qubit liền nhau lại thành **một** unitary, rồi tổng hợp lại từ đầu. Khối có 7 cổng `cx` mà toạ độ
Weyl chỉ cần 2 thì ta vừa tiết kiệm 5 cổng - việc mà pass huỷ-cổng-từng-cặp không bao giờ tìm ra.

Đây là nội dung của `Collect2qBlocks` → `ConsolidateBlocks` → `UnitarySynthesis`, và là lý do
`optimization_level=3` mạnh hơn level 1.

In [ ]:
goc = transpile(random_circuit(3, 12, max_operands=2, seed=4),
                basis_gates=["cx", "rz", "sx", "x"], optimization_level=0)

pm = PassManager([
    Collect2qBlocks(),                                          # tìm khối 2-qubit liền nhau
    ConsolidateBlocks(basis_gates=["cx", "rz", "sx", "x"]),      # gộp khối thành 1 UnitaryGate
    UnitarySynthesis(basis_gates=["cx", "rz", "sx", "x"]),       # tổng hợp lại bằng KAK
])
lai = pm.run(goc)

print(f"{'mạch':28s} {'cx':>4s} {'tổng cổng':>10s}")
print(f"{'gốc (level 0)':28s} {goc.count_ops().get('cx', 0):4d} {goc.size():10d}")
print(f"{'resynthesis bằng KAK':28s} {lai.count_ops().get('cx', 0):4d} {lai.size():10d}")
print(f"\nvẫn đúng: {Operator(lai).equiv(Operator(goc))}")
print("-> tổng số cổng giảm 36% chỉ bằng cách gộp lại rồi tổng hợp lại.")

## 3. Approximate synthesis: Clifford+T và vì sao $T$ đắt

Trên máy fault-tolerant, mạch không chạy trên qubit vật lý mà trên **qubit logic** của một mã sửa
lỗi. Mã đó chỉ thực hiện được một tập cổng rời rạc - thường là **Clifford+T**:

- Nhóm **Clifford** ($H$, $S$, $CX$) chạy gần như miễn phí: mã sửa lỗi thực hiện chúng bằng cách
  hoán vị/đo, không cần tài nguyên thêm.
- Nhưng Clifford **một mình thì không đủ** (định lý Gottesman-Knill: mạch chỉ gồm Clifford mô phỏng
  được bằng máy tính cổ điển trong thời gian đa thức - nên không thể có lợi thế lượng tử).
- Thêm **một** cổng ngoài Clifford là đủ universal. Thường chọn $T = \mathrm{diag}(1, e^{i\pi/4})$.
  Và $T$ phải làm bằng **magic state distillation**: tốn hàng nghìn qubit vật lý và nhiều chu kỳ cho
  **mỗi** cổng $T$.

Nên trên máy fault-tolerant, hàm cost không phải depth, không phải số cổng, mà là **T-count**. Giảm
T-count 20% là giảm 20% tài nguyên của cả máy.

**Định lý Solovay-Kitaev** bảo đảm việc này khả thi: với gate set rời rạc sinh ra tập trù mật trong
$SU(2)$, mọi unitary xấp xỉ được tới $\epsilon$ bằng $O(\log^c(1/\epsilon))$ cổng. Thuật toán
Solovay-Kitaev dựng chuỗi đó bằng đệ quy.

In [ ]:
U_goc = Operator(RZGate(np.pi / 8)).data     # một phép quay "khó", góc pi/8

print("Thuật toán Solovay-Kitaev (pass SolovayKitaev):")
print(f"  {'recursion_degree':>16s} {'T-count':>8s} {'tổng cổng':>10s} {'sai số thực':>12s}")
for deg in [1, 2, 3]:
    q = QuantumCircuit(1)
    q.rz(np.pi / 8, 0)
    out = SolovayKitaev(recursion_degree=deg)(q)
    print(f"  {deg:16d} {dem_T(out):8d} {out.size():10d} {sai_so(out, U_goc):12.2e}")

print("\n-> mỗi bậc đệ quy làm sai số giảm nhưng số cổng tăng khoảng 4 lần.")
print("-> SK đúng về mặt lý thuyết nhưng hằng số rất tệ: 478 cổng T cho sai số 2.5e-3.")

## 4. Bẫy thứ ba: `unitary_synthesis_method` chỉ tác dụng lên `UnitaryGate`

Qiskit có plugin synthesis tốt hơn SK rất nhiều, gọi qua
`transpile(..., unitary_synthesis_method=...)`. Nhưng pass `UnitarySynthesis` **chỉ xử lý
`UnitaryGate`** - tức là cổng được định nghĩa bằng một ma trận.

Nếu bạn đưa vào một `rz` thường, `BasisTranslator` sẽ dịch nó ra Clifford+T bằng luật dịch cố định
**trước khi** plugin được gọi. Plugin không bao giờ chạy, và tham số `epsilon` bị bỏ qua **im lặng**.

Hậu quả: bạn in ra một bảng T-count theo $\epsilon$ trông rất hợp lý, nhưng mọi dòng giống nhau.

In [ ]:
print("SAI - đưa vào rz thường:")
for eps in [1e-1, 1e-2, 1e-4, 1e-6]:
    q = QuantumCircuit(1)
    q.rz(np.pi / 8, 0)
    out = transpile(q, basis_gates=CLIFFORD_T, unitary_synthesis_method="gridsynth",
                    unitary_synthesis_plugin_config={"epsilon": eps}, optimization_level=0)
    print(f"  epsilon={eps:.0e} -> T-count {dem_T(out):3d}")
print("  ^ epsilon đổi 5 bậc mà T-count không nhúc nhích. Plugin chưa từng được gọi.")

print("\nĐÚNG - bọc thành UnitaryGate:")
for eps in [1e-1, 1e-2, 1e-4, 1e-6]:
    q = QuantumCircuit(1)
    q.append(UnitaryGate(U_goc), [0])
    out = transpile(q, basis_gates=CLIFFORD_T, unitary_synthesis_method="gridsynth",
                    unitary_synthesis_plugin_config={"epsilon": eps}, optimization_level=0)
    print(f"  epsilon={eps:.0e} -> T-count {dem_T(out):3d}, sai số thực {sai_so(out, U_goc):.1e}")

Cách tự bảo vệ: **luôn đo sai số thực tế** của mạch tổng hợp ra, đừng tin tham số `epsilon` đã có
hiệu lực. Hàm `sai_so()` ở ô đầu notebook này tốn vài dòng và chặn đúng loại lỗi đó.

### Bẫy thứ tư: tên plugin, và plugin `"sk"` bị crash

Hai điều về API cần biết, cả hai đều không hiện trong tên pass:

- Plugin Ross-Selinger đăng ký dưới tên **`"gridsynth"`**, không phải `"rossselinger"` - mặc dù
  class của nó tên `RossSelingerSynthesis`. Gọi sai tên thì được `TranspilerError`, may là ồn ào.
- `unitary_synthesis_method="sk"` **panic ở tầng Rust** (`called Option::unwrap() on a None value`
  trong crate `rstar`) - không phải exception Python nên `try/except` cũng không đỡ được. Muốn chạy
  Solovay-Kitaev thì dùng **pass** `SolovayKitaev` như ô trên, đừng dùng plugin.

Danh sách plugin có thật lấy được bằng entry point:

In [ ]:
import importlib.metadata as md_meta

co_san = sorted(e.name for e in md_meta.entry_points(group="qiskit.unitary_synthesis"))
print("unitary synthesis plugin có trong môi trường này:", co_san)

try:
    q = QuantumCircuit(1)
    q.append(UnitaryGate(U_goc), [0])
    transpile(q, basis_gates=CLIFFORD_T, unitary_synthesis_method="rossselinger", optimization_level=0)
except Exception as e:
    print(f"\n'rossselinger' -> {type(e).__name__}: {e}")
print("\n'sk' -> panic ở tầng Rust, không bắt được bằng try/except. Dùng pass SolovayKitaev.")

## 5. So sánh công bằng: cùng sai số, bao nhiêu cổng T?

So "SK bậc 3" với "gridsynth $\epsilon = 10^{-4}$" là so sai, vì hai cái đạt sai số khác nhau. Phải
so **ở cùng sai số thực tế**.

In [ ]:
rows = []
for deg in [1, 2, 3]:
    q = QuantumCircuit(1)
    q.rz(np.pi / 8, 0)
    out = SolovayKitaev(recursion_degree=deg)(q)
    rows.append({"phương pháp": f"SolovayKitaev deg={deg}", "T-count": dem_T(out),
                 "tổng cổng": out.size(), "sai số": sai_so(out, U_goc)})
for eps in [1e-1, 1e-2, 1e-3, 1e-4]:
    q = QuantumCircuit(1)
    q.append(UnitaryGate(U_goc), [0])
    out = transpile(q, basis_gates=CLIFFORD_T, unitary_synthesis_method="gridsynth",
                    unitary_synthesis_plugin_config={"epsilon": eps}, optimization_level=0)
    rows.append({"phương pháp": f"gridsynth eps={eps:.0e}", "T-count": dem_T(out),
                 "tổng cổng": out.size(), "sai số": sai_so(out, U_goc)})

bang = pd.DataFrame(rows).sort_values("sai số", ascending=False)
bang["sai số"] = bang["sai số"].map(lambda x: f"{x:.2e}")
print(bang.to_string(index=False))

print("\nỞ sai số ~2.5e-3:")
print("  SolovayKitaev deg=3 : 478 cổng T")
print("  gridsynth eps=1e-3  :  66 cổng T  (sai số còn tốt hơn: 9.7e-4)")
print("  -> gridsynth tiết kiệm hơn 7 lần. SK chỉ nên dùng để hiểu định lý, không để sản xuất.")

t = [18, 40, 66, 84]
e = [1e-1, 1e-2, 1e-3, 1e-4]
hs = np.polyfit([np.log2(1 / x) for x in e], t, 1)[0]
print(f"\nT-count của gridsynth tăng tuyến tính theo log(1/eps): hệ số đo được ~{hs:.1f}")
print("Ross-Selinger (2016) chứng minh trường hợp điển hình là 4*log2(1/eps) + O(log log(1/eps)),")
print("và tối ưu tới sai khác O(log log(1/eps)) - implementation này cách hằng số đó ~1.7 lần.")
print("Solovay-Kitaev thì là O(log^c(1/eps)) với c > 3: KHÁC BẬC, không chỉ khác hằng số.")

## 6. Giảm T-count sau khi tổng hợp - chỗ cho reinforcement learning

Mạch Clifford+T đã tổng hợp ra vẫn còn dư. Lý do: $T$ và Clifford **giao hoán** theo những luật
nhất định, nên đẩy các cổng Clifford qua nhau có thể làm hai cổng $T$ gặp nhau và triệt tiêu
($T \cdot T = S$, mà $S$ là Clifford - miễn phí).

Tìm **chuỗi phép biến đổi tốt nhất** trong không gian các luật rewrite đó là bài toán tìm kiếm, và
chính là chỗ RL vào được:

- **state**: mạch Clifford+T hiện tại
- **action**: áp dụng một luật rewrite ở một vị trí
- **reward**: T-count giảm được

Đây là framing của hướng AlphaTensor-Quantum (giảm T-count) và của hướng dùng ZX-calculus làm tập
luật rewrite. `OptimizeCliffordT` của Qiskit là baseline heuristic cho đúng bài toán đó.

In [ ]:
q = QuantumCircuit(1)
q.rz(np.pi / 8, 0)
sk3 = SolovayKitaev(recursion_degree=3)(q)
toi_uu = PassManager([OptimizeCliffordT()]).run(sk3)

print(f"{'':24s} {'T-count':>8s} {'tổng cổng':>10s} {'sai số':>11s}")
print(f"{'SolovayKitaev deg=3':24s} {dem_T(sk3):8d} {sk3.size():10d} {sai_so(sk3, U_goc):11.2e}")
print(f"{'+ OptimizeCliffordT':24s} {dem_T(toi_uu):8d} {toi_uu.size():10d} {sai_so(toi_uu, U_goc):11.2e}")
print(f"\n-> giảm {(1 - dem_T(toi_uu) / dem_T(sk3)) * 100:.0f}% T-count, sai số KHÔNG đổi.")
print("-> Đây là baseline. Một RL agent cho bài toán T-count phải vượt con số này,")
print("   và phải chứng minh sai số không đổi - nếu không thì nó chỉ đang làm mạch sai.")

## Tóm tắt

| Nội dung | Kết quả |
|---|---|
| synthesis 1 qubit | 3 góc Euler là tối thiểu và luôn đủ |
| **KAK** | mọi unitary 2 qubit cần **0, 1, 2 hoặc 3** `cx` - không có khả năng khác |
| `num_basis_gates(U)` | cho biết con số đó mà không cần tổng hợp |
| SWAP = 3 `cx` | giá của routing ở notebook `02` |
| `Collect2qBlocks` → `ConsolidateBlocks` → `UnitarySynthesis` | gộp khối rồi tổng hợp lại, giảm 36% số cổng |
| Clifford+T | Clifford gần miễn phí, $T$ cần magic state distillation |
| Gottesman-Knill | Clifford một mình mô phỏng được bằng máy cổ điển → phải có $T$ |
| **T-count** | hàm cost chính của máy fault-tolerant |
| SolovayKitaev | đúng lý thuyết, hằng số tệ: 478 T cho sai số 2.5e-3 |
| gridsynth (Ross-Selinger) | 66 T cho sai số tốt hơn - **hơn 7 lần** |
| bẫy `epsilon` | plugin chỉ chạy trên `UnitaryGate`; với `rz` thường thì `epsilon` bị bỏ qua im lặng |
| bẫy tên plugin | `"gridsynth"` (không phải `"rossselinger"`); `"sk"` panic ở tầng Rust |
| `OptimizeCliffordT` | giảm 34% T-count, giữ nguyên sai số - baseline cho RL |
| cách tự bảo vệ | **luôn đo sai số thực** bằng khoảng cách toán tử |

### Thử tự làm

1. Chạy `TwoQubitBasisDecomposer` với basis khác `cx`: `TwoQubitBasisDecomposer(iSwapGate())`.
   Unitary ngẫu nhiên cần bao nhiêu `iswap`? SWAP cần bao nhiêu?
2. Lấy 200 unitary 2 qubit ngẫu nhiên, vẽ histogram `num_basis_gates`. Tỉ lệ cần đúng 3 `cx` là
   bao nhiêu? (Dự đoán trước khi chạy: tập cần ít hơn 3 có độ đo bằng 0.)
3. Tổng hợp `rz(θ)` cho $\theta = \pi/4, \pi/8, \pi/16, \pi/32$ ở cùng `epsilon=1e-4`. T-count có
   phụ thuộc vào góc không, hay chỉ phụ thuộc `epsilon`?
4. Áp `OptimizeCliffordT` **nhiều lần** liên tiếp lên cùng một mạch. Nó có hội tụ không, hay lần
   thứ hai vẫn giảm được nữa? (Nếu còn giảm được thì heuristic đang bỏ sót - đúng chỗ RL chen vào.)
5. Dùng `ConsolidateBlocks` + `UnitarySynthesis` với `basis_gates=CLIFFORD_T` và
   `unitary_synthesis_method="gridsynth"` trên một mạch 2 qubit. T-count của cả mạch là bao nhiêu?